# 02 – Evaluate the saved model

Loads the joblib file from `MODEL_PATH` and scores the 2025 test rows that already have a next-quarter actual. Does not retrain.

In [0]:
import joblib
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from configs.project_config import (
    GOLD_TABLES,
    MODEL_PATH,
    TEST_YEAR,
    NUMERIC_FEATURES,
    BOOLEAN_FEATURES,
    CATEGORICAL_FEATURES,
    LABEL_COL,
)

feature_cols = NUMERIC_FEATURES + BOOLEAN_FEATURES + CATEGORICAL_FEATURES

In [0]:
saved = joblib.load(MODEL_PATH)
model = saved["model"]
print(f"Loaded {saved['model_name']} from {MODEL_PATH}")

In [0]:
pdf = (
    spark.table(GOLD_TABLES["ml_crop_forecast_features"])
    .where(f"year = {TEST_YEAR} AND {LABEL_COL} IS NOT NULL")
    .toPandas()
)
for col in BOOLEAN_FEATURES:
    pdf[col] = pdf[col].astype("float")
if pdf.empty:
    raise ValueError(f"No labeled rows for test year {TEST_YEAR}.")

predicted = np.asarray(model.predict(pdf[feature_cols]), dtype=float)
actual = pdf[LABEL_COL].to_numpy(dtype=float)
error = predicted - actual
nonzero = actual != 0

metrics = {
    "mae": float(mean_absolute_error(actual, predicted)),
    "rmse": float(np.sqrt(mean_squared_error(actual, predicted))),
    "mape": float(np.mean(np.abs((actual[nonzero] - predicted[nonzero]) / actual[nonzero]))) if nonzero.any() else None,
    "r2": float(r2_score(actual, predicted)),
    "rows": int(len(pdf)),
}
print(f"Test year {TEST_YEAR}")
for key, value in metrics.items():
    print(f"  {key}: {value}")

In [0]:
scored = pdf[["province", "crop", "crop_group", "year", "quarter"]].copy()
scored["actual_production"] = actual
scored["predicted_production"] = predicted
scored["error"] = error

by_group = (
    scored.groupby("crop_group")[["actual_production", "predicted_production"]]
    .apply(lambda frame: mean_absolute_error(frame["actual_production"], frame["predicted_production"]))
    .rename("mae")
    .reset_index()
)
print("MAE by crop group")
print(by_group.to_string(index=False))
print("Next: 03_batch_inference")